# portfolio_analysis — Pandas 평가/손익

- 담당: (이름)  브랜치: `<본인 github id>`
- 규칙: 커밋 전 `Kernel → Restart & Clear Output`. 이 노트북은 1인 전용.


In [ ]:
import sys, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from backend.services.portfolio.calculator import load_data, holdings, valuation, position_pnl, top_bottom
pf, sp = load_data(ROOT)            # 원본 컬럼(date/stock_id/price…)을 표준명으로 매핑해서 읽음
pf.head(), sp.head()

## 1. 데이터 이해 (컬럼, 결측, 중복)

In [ ]:
print(pf.info()); print(sp.info())
print("결측:", pf.isna().sum().sum(), sp.isna().sum().sum(), " 중복:", pf.duplicated().sum(), sp.duplicated().sum())
print("포지션", len(pf), "종목", pf.ticker.nunique(), "| 시세 종목", sp.ticker.nunique(), "거래일", sp.trade_date.nunique(), sp.trade_date.min().date(), "~", sp.trade_date.max().date())
holdings(pf).sort_values("positions", ascending=False).head()

## 2. 종목별 평가금액 / 손익 / 손익률 / 비중 — 포지션을 종목별로 SUM (SQL [A-2]와 동일 계산)

In [ ]:
val = valuation(pf, sp)
val

## 3. 상위/하위 3 종목

In [ ]:
top3, bottom3 = val.head(3), val.tail(3)
pos = position_pnl(pf, sp)          # 포지션 단위: 가장 잘/못 산 매수 건 (SQL [A-4])
top3, bottom3, pos.head(5), pos.tail(5)

## 4. 결과 내보내기 (검증 담당이 읽는 파일)

In [ ]:
out = ROOT/"outputs"; out.mkdir(exist_ok=True)
val.to_csv(out/"pandas_C_valuation.csv", index=False)
top_bottom(val).to_csv(out/"pandas_C_top_bottom.csv", index=False)
pos.to_csv(out/"pandas_C_positions.csv", index=False)